# Train

## Imports

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


## Params

In [ ]:
_cwd = Path.cwd()
ROOT = _cwd if (_cwd / 'scripts').exists() else _cwd.parent
DATA_DIR = ROOT / 'data' / 'hand_poses_public'
CKPT = ROOT / 'models' / 'digits_mlp.pth'

EPOCHS = 200
LR = 1e-3
BATCH_SIZE = 64
HIDDEN = 128
DROPOUT = 0.3
VAL_FRAC = 0.2
PATIENCE = 30
SEED = 42


## Model

In [ ]:
class HandMLP(nn.Module):
    def __init__(self, num_classes, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, hidden), nn.ReLU(),
            nn.BatchNorm1d(hidden), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.BatchNorm1d(hidden), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)

def augment_batch(x):
    b, d = x.shape
    x = x + torch.randn_like(x) * 0.01
    scale = torch.empty(b, 1, device=x.device).uniform_(0.95, 1.05)
    x = x * scale
    angles = torch.empty(b, device=x.device).uniform_(-0.175, 0.175)
    c = torch.cos(angles)
    s = torch.sin(angles)
    x3 = x.view(b, 21, 3).clone()
    rx = c.view(b, 1) * x3[:, :, 0] - s.view(b, 1) * x3[:, :, 1]
    ry = s.view(b, 1) * x3[:, :, 0] + c.view(b, 1) * x3[:, :, 1]
    x3[:, :, 0] = rx
    x3[:, :, 1] = ry
    return x3.view(b, d)

def run_epoch(model, loader, device, crit, opt=None, augment=False):
    train = opt is not None
    model.train() if train else model.eval()
    loss_sum = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train and augment:
                x = augment_batch(x)
            logits = model(x)
            loss = crit(logits, y)
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
            loss_sum += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


## Data

In [ ]:
X = np.load(DATA_DIR / 'X.npy').astype(np.float32)
y = np.load(DATA_DIR / 'y.npy').astype(np.int64)
labels = json.loads((DATA_DIR / 'labels.json').read_text())
num_classes = max(int(y.max()) + 1, len(labels))
print(f'Loaded {X.shape[0]} samples across {num_classes} classes')
for c in range(num_classes):
    name = labels.get(str(c), str(c))
    print(f'  {name:>6}: {int((y == c).sum()):>4}')


## Setup

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
idx = rng.permutation(X.shape[0])
val_n = max(int(X.shape[0] * VAL_FRAC), num_classes)
val_idx, tr_idx = idx[:val_n], idx[val_n:]
mean = X[tr_idx].mean(0)
std = X[tr_idx].std(0)
std[std < 1e-6] = 1.0
Xn = (X - mean) / std

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tr_ds = TensorDataset(torch.from_numpy(Xn[tr_idx]).float(), torch.from_numpy(y[tr_idx]).long())
vl_ds = TensorDataset(torch.from_numpy(Xn[val_idx]).float(), torch.from_numpy(y[val_idx]).long())
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False)

model = HandMLP(num_classes, hidden=HIDDEN, dropout=DROPOUT).to(device)
crit = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
print(f'Train: {len(tr_idx)}  Val: {len(val_idx)}  Params: {sum(p.numel() for p in model.parameters()):,}')


## Train

In [ ]:
CKPT.parent.mkdir(parents=True, exist_ok=True)
best = -1.0
bad = 0
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    tl, ta = run_epoch(model, tr_loader, device, crit, opt, augment=True)
    vl, va = run_epoch(model, vl_loader, device, crit)
    print(f'Ep {ep:3d}  tr {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}')
    if va > best:
        best = va
        bad = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'hidden': HIDDEN, 'dropout': DROPOUT,
            'num_classes': num_classes,
            'mean': mean.astype(np.float32),
            'std': std.astype(np.float32),
            'labels': labels,
            'best_val_acc': best, 'epoch': ep,
        }, CKPT)
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'Early stop at ep{ep}')
            break
print(f'Best val acc: {best*100:.1f}%   time: {time.time()-t0:.1f}s')
print(f'Saved to {CKPT}')
